In [ ]:
import pandas as pd
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.metrics import adjusted_rand_score

#load the Datasets
clinical_df_train = pd.read_csv('../data/processed/clinical_kirc_train_clustered_pca.csv')
methylation_df_train = pd.read_parquet('../data/processed/methylation_kirc_train.parquet')
clinical_df_holdout = pd.read_csv('../data/processed/clinical_kirc_holdout.csv')
methylation_df_holdout = pd.read_parquet('../data/processed/methylation_kirc_holdout.parquet')

#align both clinical dataframes to their corresponding methylation dataframe's row order by sample ID
clinical_df_train = clinical_df_train.set_index('sample').reindex(methylation_df_train.index)
clinical_df_holdout = clinical_df_holdout.set_index('sample').reindex(methylation_df_holdout.index)


In [2]:
loaded_pca = joblib.load('../models/pca_best_model.joblib')

loaded_clustering = joblib.load('../models/clustering_best_model.joblib')


In [ ]:
#extract sample IDs from the index
meth_samples_train = methylation_df_train.index.values

#extract the raw methylation features (all columns)
meth_features_train = methylation_df_train.values

#scale the data before PCA and Clustering
print("Scaling data...")
scaler1 = StandardScaler()
meth_features_train_scaled = scaler1.fit_transform(meth_features_train)

#use the SCALED features; the saved PCA model was fit on scaled data in 02-1_clustering_pca.ipynb, so transforming raw (unscaled) features here produced meaningless components
X_train_reduced = loaded_pca.transform(meth_features_train_scaled)

cluster_labels_train = loaded_clustering.predict(X_train_reduced)

Scaling data...


/Users/matibai-tcm/Library/Python/3.9/lib/python/site-packages/sklearn/decomposition/_base.py:148: RuntimeWarning: divide by zero encountered in matmul
  X_transformed = X @ self.components_.T
/Users/matibai-tcm/Library/Python/3.9/lib/python/site-packages/sklearn/decomposition/_base.py:148: RuntimeWarning: overflow encountered in matmul
  X_transformed = X @ self.components_.T
/Users/matibai-tcm/Library/Python/3.9/lib/python/site-packages/sklearn/decomposition/_base.py:148: RuntimeWarning: invalid value encountered in matmul
  X_transformed = X @ self.components_.T
/Users/matibai-tcm/Library/Python/3.9/lib/python/site-packages/sklearn/decomposition/_base.py:155: RuntimeWarning: divide by zero encountered in matmul
  X_transformed -= xp.reshape(self.mean_, (1, -1)) @ self.components_.T
/Users/matibai-tcm/Library/Python/3.9/lib/python/site-packages/sklearn/decomposition/_base.py:155: RuntimeWarning: overflow encountered in matmul
  X_transformed -= xp.reshape(self.mean_, (1, -1)) @ self.

In [ ]:
#extract sample IDs from the index
meth_samples_holdout = methylation_df_holdout.index.values

#extract the raw methylation features (all columns)
meth_features_holdout = methylation_df_holdout.values

#reuse the SAME scaler fit on training data (scaler1) instead of fitting a new scaler on holdout -- fitting preprocessing on holdout leaks its distribution into evaluation, the same class of bug as the PCA-reduction fix in 04_classification.ipynb
print("Scaling data...")
meth_features_holdout_scaled = scaler1.transform(meth_features_holdout)

X_holdout_reduced = loaded_pca.transform(meth_features_holdout_scaled)

cluster_labels_holdout = loaded_clustering.predict(X_holdout_reduced)

Scaling data...


/Users/matibai-tcm/Library/Python/3.9/lib/python/site-packages/sklearn/decomposition/_base.py:148: RuntimeWarning: divide by zero encountered in matmul
  X_transformed = X @ self.components_.T
/Users/matibai-tcm/Library/Python/3.9/lib/python/site-packages/sklearn/decomposition/_base.py:148: RuntimeWarning: overflow encountered in matmul
  X_transformed = X @ self.components_.T
/Users/matibai-tcm/Library/Python/3.9/lib/python/site-packages/sklearn/decomposition/_base.py:148: RuntimeWarning: invalid value encountered in matmul
  X_transformed = X @ self.components_.T
/Users/matibai-tcm/Library/Python/3.9/lib/python/site-packages/sklearn/decomposition/_base.py:155: RuntimeWarning: divide by zero encountered in matmul
  X_transformed -= xp.reshape(self.mean_, (1, -1)) @ self.components_.T
/Users/matibai-tcm/Library/Python/3.9/lib/python/site-packages/sklearn/decomposition/_base.py:155: RuntimeWarning: overflow encountered in matmul
  X_transformed -= xp.reshape(self.mean_, (1, -1)) @ self.

In [5]:
ari_score_train = adjusted_rand_score(clinical_df_train['tissue_type.samples'], cluster_labels_train)
print(f"Train : {ari_score_train:.4f}")
ari_score_holdout = adjusted_rand_score(clinical_df_holdout['tissue_type.samples'], cluster_labels_holdout)
print(f"Holdout : {ari_score_holdout:.4f}")  #ClaudeCode Ver. 1: this was mislabeled "Train" before

Train : 0.8760
Holdout : 0.8245
